# XAI Colab RAG Reranker ver.


## RAG 구조

1. 사용자 질문 입력
2. Qdrant dense vector search로 후보 문서 넉넉히 검색
3. `Dongjin-kr/ko-reranker`로 질문-문서 관련성 재평가
4. 상위 문서만 최종 context로 구성
5. Qwen 모델이 정책 상담 답변 생성

## 참고 자료
https://aws.amazon.com/ko/blogs/tech/korean-reranker-rag/

https://devocean.sk.com/blog/techBoardDetail.do?ID=167335&boardType=techBlog


## 설계 의도

단순 벡터 검색 top-k를 그대로 사용하는 방식은 질문의 세부 의도와 가장 관련 높은 문서를 항상 상단에 배치하지 못할 수 있다.  
따라서 본 서버는 1차 검색에서 recall을 확보하고, 2차 reranker에서 precision을 높이는 2-stage RAG 구조를 사용한다.

###환경

- LLM: `Qwen/Qwen3-4B-Instruct-2507`
- Embedding: `BAAI/bge-m3`
- Reranker: `Dongjin-kr/ko-reranker`
- Vector DB: Qdrant
- API Server: FastAPI
- Tunnel: Cloudflare Tunnel

In [ ]:
# ============================================================
# 1. Drive 연결 및 실험용 작업 경로 설정
# ============================================================
# 기존 타 경로와 충돌하지 않도록
# 실험용 폴더(MyProject_test1)를 별도로 사용한다.
# ============================================================

import os
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

# 작업 폴더
PROJECT_PATH = Path("/content/drive/MyDrive/MyProject_test1")
PROJECT_PATH.mkdir(parents=True, exist_ok=True)

os.chdir(PROJECT_PATH)

# 캐시/로그/모델 경로
HF_HOME = PROJECT_PATH / "hf_cache"
LOCAL_MODEL_DIR = PROJECT_PATH / "local_models"
LOG_DIR = PROJECT_PATH / "logs"

HF_HOME.mkdir(parents=True, exist_ok=True)
LOCAL_MODEL_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(HF_HOME)

print("작업 위치:", PROJECT_PATH)
print("HF cache:", HF_HOME)
print("Local model dir:", LOCAL_MODEL_DIR)
print("Log dir:", LOG_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
작업 위치: /content/drive/MyDrive/MyProject_test1
HF cache: /content/drive/MyDrive/MyProject_test1/hf_cache
Local model dir: /content/drive/MyDrive/MyProject_test1/local_models
Log dir: /content/drive/MyDrive/MyProject_test1/logs


In [ ]:
# ============================================================
# 2. 라이브러리 설치
# ============================================================

# 기본 RAG / LLM / Reranker / 서버 실행에 필요한 패키지
!pip install -q -U \
    transformers \
    accelerate \
    sentence-transformers \
    qdrant-client \
    fastapi \
    uvicorn \
    pydantic \
    nest-asyncio \
    captum

# Cloudflare Tunnel 실행 파일 설치
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb -O cloudflared.deb
!dpkg -i cloudflared.deb > /dev/null 2>&1

# 설치 확인
!cloudflared --version

print("라이브러리 설치 완료")

cloudflared version 2026.3.0 (built 2026-03-09-14:08 UTC)
라이브러리 설치 완료


In [ ]:
# ============================================================
# 3. 설정값 로드 및 기본 설정
# ============================================================
# 이 셀은 모델 ID, Qdrant 접속 정보, RAG 검색 개수,
# Reranker 모델 ID 등을 한 곳에서 관리한다.
#
# [설계 의도]
# - API KEY 같은 민감정보는 코드에 직접 쓰지 않고 Colab userdata에서 읽는다.
# - 모델/검색 개수는 실험 중 자주 바뀔 수 있으므로 설정값으로 분리한다.
# - 교수님/팀원 공유 시 어떤 외부 모델을 사용했는지 명확히 확인할 수 있도록 한다.
# ============================================================

import os
from google.colab import userdata


# ------------------------------------------------------------
# Colab userdata → 환경변수 → 기본값 순서로 설정값을 읽는 함수
# ------------------------------------------------------------
def get_setting(key: str, default: str = "") -> str:
    """
    설정값을 읽는 공통 함수.

    우선순위:
    1. Colab userdata
    2. 환경변수(os.environ)
    3. 코드에 지정한 기본값
    """
    try:
        value = userdata.get(key)
        if value is not None and str(value).strip():
            return str(value).strip()
    except Exception:
        pass

    value = os.getenv(key)
    if value is not None and str(value).strip():
        return str(value).strip()

    return default

# ============================================================
# Hugging Face Token 설정
# ============================================================

from huggingface_hub import login

HF_TOKEN = get_setting("HF_TOKEN", "")

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    login(token=HF_TOKEN, add_to_git_credential=False)
    print("Hugging Face token 설정 완료")
else:
    print("[WARN] HF_TOKEN이 설정되지 않았습니다. 공개 모델은 다운로드 가능하지만 rate limit 경고가 발생할 수 있습니다.")


def get_int_setting(key: str, default: int) -> int:
    """
    정수형 설정값을 읽는 함수.
    예: QDRANT_FETCH_K, RERANK_TOP_N, FINAL_TOP_K
    """
    value = get_setting(key, str(default))
    try:
        return int(value)
    except ValueError:
        print(f"[WARN] {key}={value} 값을 int로 변환할 수 없어 기본값 {default} 사용")
        return default


def get_float_setting(key: str, default: float) -> float:
    """
    실수형 설정값을 읽는 함수.
    예: GEN_TEMPERATURE, GEN_TOP_P
    """
    value = get_setting(key, str(default))
    try:
        return float(value)
    except ValueError:
        print(f"[WARN] {key}={value} 값을 float로 변환할 수 없어 기본값 {default} 사용")
        return default


# ============================================================
#  LLM 설정
# ============================================================

MODEL_ID = get_setting("MODEL_ID", "Qwen/Qwen3-4B-Instruct-2507")


# ============================================================
# Embedding 모델 설정
# ============================================================
# - BAAI/bge-m3 사용
# - 역할: 사용자 질문을 벡터로 변환하여 Qdrant dense retrieval 수행
# - bge-m3는 다국어 검색에 활용 가능한 embedding 모델
# ============================================================

EMBED_MODEL_ID = get_setting("EMBED_MODEL_ID", "BAAI/bge-m3")


# ============================================================
# [외부 공개 모델 활용] Korean Reranker 설정
# ============================================================
# - Dongjin-kr/ko-reranker 사용
# - 역할: Qdrant 1차 검색 후보를 질문-문서 관련성 기준으로 재정렬
# - MIT License로 공개된 한국어 reranker
#
# [RAG 설계 근거]
# - 1차 dense retrieval은 후보 문서를 넓게 가져오는 recall 확보 단계
# - 2차 reranking은 질문과 문서를 pair로 비교하여 precision을 높이는 단계
# ============================================================

RERANKER_MODEL_ID = get_setting("RERANKER_MODEL_ID", "Dongjin-kr/ko-reranker")


# ============================================================
#  Qdrant 설정
# ============================================================

QDRANT_URL = get_setting("QDRANT_URL", "")
QDRANT_API_KEY = get_setting("QDRANT_API_KEY", "")

# 여러 컬렉션을 쉼표로 입력 가능
# 예: welfare_ministry_bge_m3,local_government_welfare_bge_m3
QDRANT_COLLECTIONS_RAW = get_setting(
    "QDRANT_COLLECTIONS",
    "welfare_ministry_bge_m3,local_government_welfare_bge_m3"
)

QDRANT_COLLECTIONS = [
    name.strip()
    for name in QDRANT_COLLECTIONS_RAW.split(",")
    if name.strip()
]

# named vector를 쓰는 경우에만 설정
# 일반 단일 벡터 컬렉션이면 빈 값으로 둔다.
QDRANT_VECTOR_NAME = get_setting("QDRANT_VECTOR_NAME", "").strip() or None

# ============================================================
# RAG 검색 설정
# ============================================================
# - QDRANT_FETCH_K: 1차 dense retrieval 후보 수
# - RERANK_TOP_N: reranker 재정렬 후 남길 후보 수
# - FINAL_TOP_K: LLM context에 최종 투입할 정책 수
#
# 기본 전략:
# 1. Qdrant에서 후보를 넉넉히 가져온다.  예: 30개
# 2. Reranker로 질문 관련성을 재평가한다. 예: 상위 10개
# 3. 중복 제거 후 최종 3~5개만 LLM에 넣는다.
# ============================================================

QDRANT_FETCH_K = get_int_setting("QDRANT_FETCH_K", 30)
RERANK_TOP_N = get_int_setting("RERANK_TOP_N", 10)
FINAL_TOP_K = get_int_setting("FINAL_TOP_K", 5)

MAX_CONTEXT_CHARS_PER_POLICY = get_int_setting("MAX_CONTEXT_CHARS_PER_POLICY", 900)


# ============================================================
# 답변 생성 설정
# ============================================================

MAX_NEW_TOKENS = get_int_setting("MAX_NEW_TOKENS", 700)
GEN_TEMPERATURE = get_float_setting("GEN_TEMPERATURE", 0.3)
GEN_TOP_P = get_float_setting("GEN_TOP_P", 0.9)
GEN_REPETITION_PENALTY = get_float_setting("GEN_REPETITION_PENALTY", 1.1)


# ============================================================
# 설정 확인 출력
# ============================================================

print("========== 설정 확인 ==========")
print(f"MODEL_ID              : {MODEL_ID}")
print(f"EMBED_MODEL_ID        : {EMBED_MODEL_ID}")
print(f"RERANKER_MODEL_ID     : {RERANKER_MODEL_ID}")
print(f"QDRANT_URL 설정 여부  : {'OK' if QDRANT_URL else 'MISSING'}")
print(f"QDRANT_API_KEY 여부   : {'OK' if QDRANT_API_KEY else 'MISSING'}")
print(f"QDRANT_COLLECTIONS    : {QDRANT_COLLECTIONS}")
print(f"QDRANT_VECTOR_NAME    : {QDRANT_VECTOR_NAME}")
print(f"QDRANT_FETCH_K        : {QDRANT_FETCH_K}")
print(f"RERANK_TOP_N          : {RERANK_TOP_N}")
print(f"FINAL_TOP_K           : {FINAL_TOP_K}")
print(f"MAX_NEW_TOKENS        : {MAX_NEW_TOKENS}")
print("================================")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face token 설정 완료
========== 설정 확인 ==========
MODEL_ID              : Qwen/Qwen3-4B-Instruct-2507
EMBED_MODEL_ID        : BAAI/bge-m3
RERANKER_MODEL_ID     : Dongjin-kr/ko-reranker
QDRANT_URL 설정 여부  : OK
QDRANT_API_KEY 여부   : OK
QDRANT_COLLECTIONS    : ['welfare_ministry_bge_m3', 'local_government_welfare_bge_m3']
QDRANT_VECTOR_NAME    : None
QDRANT_FETCH_K        : 30
RERANK_TOP_N          : 10
FINAL_TOP_K           : 5
MAX_NEW_TOKENS        : 400


In [ ]:
# ============================================================
# 4. 필수 설정 검증 및 로컬 경로 설정
# ============================================================

from pathlib import Path


# ------------------------------------------------------------
# 필수 설정 검증
# ------------------------------------------------------------
missing_settings = []

if not QDRANT_URL:
    missing_settings.append("QDRANT_URL")

if not QDRANT_API_KEY:
    missing_settings.append("QDRANT_API_KEY")

if not QDRANT_COLLECTIONS:
    missing_settings.append("QDRANT_COLLECTIONS")

if missing_settings:
    raise RuntimeError(
        "필수 설정이 누락되었습니다: "
        + ", ".join(missing_settings)
        + "\nColab userdata 또는 환경변수에 값을 설정해 주세요."
    )


# ------------------------------------------------------------
# 경로 설정
# ------------------------------------------------------------

LOCAL_LLM_PATH = LOCAL_MODEL_DIR / "qwen3-4b-instruct-2507"
LOCAL_EMBED_PATH = LOCAL_MODEL_DIR / "bge-m3"
LOCAL_RERANKER_PATH = LOCAL_MODEL_DIR / "ko-reranker"

RAG_LOG_PATH = LOG_DIR / "rag_debug.log"


# 경로 생성
LOCAL_LLM_PATH.mkdir(parents=True, exist_ok=True)
LOCAL_EMBED_PATH.mkdir(parents=True, exist_ok=True)
LOCAL_RERANKER_PATH.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# 설정 확인 출력
# ------------------------------------------------------------
print("========== 4단계 설정 검증 완료 ==========")
print("Qdrant URL         :", "OK")
print("Qdrant API Key     :", "OK")
print("Qdrant Collections :", QDRANT_COLLECTIONS)
print("LLM local path     :", LOCAL_LLM_PATH)
print("Embed local path   :", LOCAL_EMBED_PATH)
print("Reranker path      :", LOCAL_RERANKER_PATH)
print("RAG log path       :", RAG_LOG_PATH)
print("==========================================")

========== 4단계 설정 검증 완료 ==========
Qdrant URL         : OK
Qdrant API Key     : OK
Qdrant Collections : ['welfare_ministry_bge_m3', 'local_government_welfare_bge_m3']
LLM local path     : /content/drive/MyDrive/MyProject_test1/local_models/qwen3-4b-instruct-2507
Embed local path   : /content/drive/MyDrive/MyProject_test1/local_models/bge-m3
Reranker path      : /content/drive/MyDrive/MyProject_test1/local_models/ko-reranker
RAG log path       : /content/drive/MyDrive/MyProject_test1/logs/rag_debug.log


In [ ]:
# ============================================================
# 5. LLM 로드
# ============================================================
# 이 셀은 최종 정책 상담 답변 생성을 담당할 LLM을 로드한다.
#
# - 역할: RAG로 구성된 정책 문서 context를 바탕으로 최종 답변 생성
# - 모델 자체를 새로 학습하지 않고, 정책 상담용 prompt와 RAG context 구성으로 답변을 제어한다.
# - 현재 Colab 환경을 전제로 하므로, 우선 양자화 없이 bf16으로 로드한다.
# ============================================================

import gc
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM


# ------------------------------------------------------------
#  디바이스 및 dtype 자동 설정
# ------------------------------------------------------------
# GPU 종류를 코드에서 강제하지 않는다.
# Colab 런타임이 A100/L4/T4 등으로 바뀌어도 자동으로 동작하도록 한다.
#
# CUDA 환경에서는 bfloat16을 사용한다.
# CPU fallback에서는 float32를 사용한다.
# ------------------------------------------------------------

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
TORCH_DTYPE = torch.bfloat16 if DEVICE == "cuda" else torch.float32

print("LLM device:", DEVICE)
print("LLM dtype :", TORCH_DTYPE)


# ------------------------------------------------------------
# 로컬 모델 파일 확인 함수
# ------------------------------------------------------------
# 이미 개인 실험 경로에 모델이 저장되어 있으면 로컬에서 로드하고,
# 없으면 Hugging Face에서 다운로드한다.
# ------------------------------------------------------------

def has_local_hf_model_files(model_path: Path) -> bool:
    """
    Hugging Face 모델이 로컬 경로에 저장되어 있는지 확인한다.
    최소한 config/tokenizer/weight 파일이 있는지 검사한다.
    """
    required_files = [
        "config.json",
        "tokenizer_config.json",
    ]

    has_required = all((model_path / filename).exists() for filename in required_files)

    has_weight = (
        (model_path / "model.safetensors").exists()
        or (model_path / "model.safetensors.index.json").exists()
        or (model_path / "pytorch_model.bin").exists()
        or (model_path / "pytorch_model.bin.index.json").exists()
    )

    return has_required and has_weight


if has_local_hf_model_files(LOCAL_LLM_PATH):
    print(f"[INFO] 로컬 LLM 경로에서 로드합니다: {LOCAL_LLM_PATH}")
    llm_source = str(LOCAL_LLM_PATH)
    local_files_only = True
else:
    print(f"[INFO] 로컬 LLM 파일이 없어 Hugging Face에서 로드합니다: {MODEL_ID}")
    llm_source = MODEL_ID
    local_files_only = False


# ------------------------------------------------------------
# Tokenizer 로드
# ------------------------------------------------------------
# Qwen 계열 tokenizer를 로드한다.
# pad_token이 없는 경우 eos_token을 pad_token으로 지정해 generation 오류를 방지한다.
# ------------------------------------------------------------

tokenizer = AutoTokenizer.from_pretrained(
    llm_source,
    trust_remote_code=True,
    local_files_only=local_files_only,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


# ------------------------------------------------------------
# 모델 로드
# ------------------------------------------------------------
# device_map="auto"를 사용해 Colab GPU에 자동 배치한다.
# 양자화 없이 bf16으로 로드하여, RAG 구조 자체의 답변 품질을 먼저 검증한다.
# ------------------------------------------------------------

model = AutoModelForCausalLM.from_pretrained(
    llm_source,
    torch_dtype=TORCH_DTYPE,
    device_map="auto" if DEVICE == "cuda" else None,
    trust_remote_code=True,
    local_files_only=local_files_only,
)

model.eval()


# ------------------------------------------------------------
#최초 다운로드 저장
# 다음 실행부터는 Hugging Face에서 다시 받지 않고 LOCAL_LLM_PATH에서 로드할 수 있다.
# ------------------------------------------------------------

if not local_files_only:
    print(f"[INFO] 모델과 tokenizer를 저장합니다: {LOCAL_LLM_PATH}")
    tokenizer.save_pretrained(str(LOCAL_LLM_PATH))
    model.save_pretrained(str(LOCAL_LLM_PATH))


# ------------------------------------------------------------
# 메모리 정리 및 로드 결과 확인
# ------------------------------------------------------------

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("========== LLM 로드 완료 ==========")
print("MODEL_ID       :", MODEL_ID)
print("LLM source     :", llm_source)
print("LOCAL_LLM_PATH :", LOCAL_LLM_PATH)
print("device         :", DEVICE)
print("dtype          :", TORCH_DTYPE)
print("pad_token      :", tokenizer.pad_token)
print("eos_token_id   :", tokenizer.eos_token_id)
print("===================================")

LLM device: cuda
LLM dtype : torch.bfloat16
[INFO] 로컬 LLM 파일이 없어 Hugging Face에서 로드합니다: Qwen/Qwen3-4B-Instruct-2507


Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

[INFO] 모델과 tokenizer를 저장합니다: /content/drive/MyDrive/MyProject_test1/local_models/qwen3-4b-instruct-2507


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

========== LLM 로드 완료 ==========
MODEL_ID       : Qwen/Qwen3-4B-Instruct-2507
LLM source     : Qwen/Qwen3-4B-Instruct-2507
LOCAL_LLM_PATH : /content/drive/MyDrive/MyProject_test1/local_models/qwen3-4b-instruct-2507
device         : cuda
dtype          : torch.bfloat16
pad_token      : <|endoftext|>
eos_token_id   : 151645


In [ ]:
# ============================================================
# 6. Embedding 모델 로드
# ============================================================
# 이 셀은 Qdrant dense retrieval에 사용할 embedding 모델을 로드한다.
#
# - BAAI/bge-m3 모델 사용
# - 역할: 사용자 질문을 dense vector로 변환하여 Qdrant에서 유사 문서 검색
#
# - 질문 embedding을 생성한 뒤, 중앙정부/지자체 Qdrant 컬렉션에서
#   후보 정책 문서를 넉넉히 검색하는 데 사용한다.
# ============================================================

from sentence_transformers import SentenceTransformer


# ------------------------------------------------------------
# Embedding 모델 로드
# 기본값: BAAI/bge-m3
# ------------------------------------------------------------

print(f"[INFO] Embedding 모델 로드 시작: {EMBED_MODEL_ID}")

embed_model = SentenceTransformer(
    EMBED_MODEL_ID,
    device=DEVICE,
)

print("========== Embedding 모델 로드 완료 ==========")
print("EMBED_MODEL_ID:", EMBED_MODEL_ID)
print("device        :", DEVICE)
print("============================================")

[INFO] Embedding 모델 로드 시작: BAAI/bge-m3


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

========== Embedding 모델 로드 완료 ==========
EMBED_MODEL_ID: BAAI/bge-m3
device        : cuda


In [ ]:
# ============================================================
# 7. Qdrant 연결
# ============================================================
# 이 셀은 정책 문서 벡터가 저장된 Qdrant DB에 연결한다.
#
# - 역할: 사용자 질문 embedding과 유사한 정책 문서 chunk 검색
# - 중앙정부 정책 컬렉션과 지자체 정책 컬렉션을 함께 검색할 수 있도록
#   QDRANT_COLLECTIONS 목록을 기준으로 연결 상태를 확인한다.
# - 이 노트북은 검색만 수행하며, upsert/delete/recreate_collection은 사용하지 않는다.
# ============================================================

from qdrant_client import QdrantClient


# ------------------------------------------------------------
# Qdrant Client 생성
# ------------------------------------------------------------

print("[INFO] Qdrant 연결을 시작합니다.")

qdrant_client = QdrantClient(
    url=QDRANT_URL,
    api_key=QDRANT_API_KEY,
)

print("[INFO] Qdrant Client 생성 완료")


# ------------------------------------------------------------
# 검색 대상 컬렉션 확인
# ------------------------------------------------------------
# get_collection()은 컬렉션 존재 여부와 기본 정보를 확인하는 용도다.
# 검색 대상 컬렉션이 없거나 이름이 틀리면 여기서 미리 확인할 수 있다.
# ------------------------------------------------------------

print("========== Qdrant 컬렉션 확인 ==========")

for collection_name in QDRANT_COLLECTIONS:
    try:
        info = qdrant_client.get_collection(collection_name)
        print(f"[OK] {collection_name}")
        print(f"     vectors_count: {getattr(info, 'vectors_count', 'unknown')}")
        print(f"     points_count : {getattr(info, 'points_count', 'unknown')}")
    except Exception as e:
        print(f"[ERROR] 컬렉션 확인 실패: {collection_name}")
        print(f"        원인: {e}")
        raise

print("=======================================")
print("[INFO] Qdrant 연결 및 컬렉션 확인 완료")

[INFO] Qdrant 연결을 시작합니다.
[INFO] Qdrant Client 생성 완료
========== Qdrant 컬렉션 확인 ==========
[OK] welfare_ministry_bge_m3
     vectors_count: unknown
     points_count : 1628
[OK] local_government_welfare_bge_m3
     vectors_count: unknown
     points_count : 4782
[INFO] Qdrant 연결 및 컬렉션 확인 완료


In [ ]:
# ============================================================
# 8. Korean Reranker 모델 로드
# ============================================================
# 이 셀은 Qdrant 1차 검색 후보를 질문-문서 관련성 기준으로
# 다시 정렬하기 위한 Reranker 모델을 로드한다.
#
# [외부 공개 모델 활용]
# - Dongjin-kr/ko-reranker 사용
# - 역할: 사용자 질문과 후보 문서를 pair로 입력받아 관련성 점수를 계산
# - MIT License로 공개된 한국어 reranker 모델
#
# [RAG 설계 근거]
# - 1차 dense retrieval은 빠르게 후보 문서를 넓게 가져오는 recall 확보 단계이다.
# - 2차 reranker는 질문과 문서를 함께 보고 관련성을 다시 평가하는 precision 향상 단계이다.
# - 최종적으로 LLM에는 reranker가 높게 평가한 문서만 context로 제공한다.
#
# - reranker 자체는 공개 모델을 활용한다.
# - 어떤 텍스트를 reranker 입력으로 만들지,
#   어떤 기준으로 중복 정책을 제거할지는 이후 셀에서 직접 구현한다.
# ============================================================

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification


# ------------------------------------------------------------
# [외부 공개 모델 활용] Reranker tokenizer/model 로드
# ------------------------------------------------------------
# RERANKER_MODEL_ID는 3번 셀에서 설정한 값이다.
# 기본값: Dongjin-kr/ko-reranker
# ------------------------------------------------------------

print(f"[INFO] Reranker 모델 로드 시작: {RERANKER_MODEL_ID}")

reranker_tokenizer = AutoTokenizer.from_pretrained(
    RERANKER_MODEL_ID,
    trust_remote_code=True,
)

reranker_model = AutoModelForSequenceClassification.from_pretrained(
    RERANKER_MODEL_ID,
    torch_dtype=TORCH_DTYPE,
    device_map="auto" if DEVICE == "cuda" else None,
    trust_remote_code=True,
)

reranker_model.eval()

print("========== Reranker 모델 로드 완료 ==========")
print("RERANKER_MODEL_ID:", RERANKER_MODEL_ID)
print("device           :", DEVICE)
print("dtype            :", TORCH_DTYPE)
print("============================================")

[INFO] Reranker 모델 로드 시작: Dongjin-kr/ko-reranker


config.json:   0%|          | 0.00/802 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/963 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

========== Reranker 모델 로드 완료 ==========
RERANKER_MODEL_ID: Dongjin-kr/ko-reranker
device           : cuda
dtype            : torch.bfloat16


In [ ]:
# ============================================================
# 9. Qdrant payload 정제 함수
# ============================================================
# 이 셀은 Qdrant 검색 결과의 payload에서 정책명, 지원내용, 지역, URL 등을 추출한다.
#
# - Qdrant에 저장된 정책 문서의 payload 구조가 항상 동일하다고 가정하지 않는다.
# - payload 직접 key, metadata 내부 key, 본문 안의 [필드명] / 필드명: 형식을 모두 확인한다.
# - Reranker 입력과 LLM context에 사용할 수 있도록 정책 정보를 정리한다.
# ============================================================

import re
from typing import Any, Dict, List, Tuple


# ------------------------------------------------------------
# payload 값 추출
# ------------------------------------------------------------
# Qdrant payload는 저장 방식에 따라 아래 두 형태가 모두 가능하다.
#
# 1. payload["정책명"]
# 2. payload["metadata"]["정책명"]
#
# 따라서 직접 key와 metadata 내부 key를 모두 확인한다.
# ------------------------------------------------------------

def get_payload_value(payload: Dict[str, Any], *keys: str) -> str:
    """
    payload에서 여러 후보 key를 순서대로 확인해 값을 추출한다.
    직접 key와 metadata 내부 key를 모두 확인한다.
    """
    if not isinstance(payload, dict):
        return ""

    candidates = []

    # payload 직접 key 확인
    for key in keys:
        if key:
            candidates.append(payload.get(key))

    # metadata 내부 key 확인
    metadata = payload.get("metadata")
    if isinstance(metadata, dict):
        for key in keys:
            if key:
                candidates.append(metadata.get(key))

    for value in candidates:
        if value is not None and str(value).strip():
            return str(value).strip()

    return ""


# ------------------------------------------------------------
# 텍스트 정규화
# ------------------------------------------------------------

def normalize_policy_text(text: str) -> str:
    """
    모델과 reranker에 넣기 전 텍스트를 간단히 정리한다.
    """
    if not text:
        return ""

    text = str(text)
    text = text.replace("、", ", ").replace("。", ". ")
    text = re.sub(r"\s+", " ", text).strip()
    return text


def clean_extracted_value(value: str) -> str:
    """
    필드 추출 결과에 다른 필드가 붙어 들어온 경우를 정리한다.
    예: '[지원내용] ... [신청방법] ...'에서 지원내용만 남기기
    """
    if not value:
        return ""

    value = normalize_policy_text(value)

    # 다음 [필드명]이 붙어 있으면 그 전까지만 사용
    value = re.split(
        r"\s+\[(?:서비스ID|서비스명|정책명|분류|분야|지역|지원유형|지원대상|지원내용|선정기준|신청정보|신청방법|상세 URL|URL|섹션)\]",
        value,
        maxsplit=1,
    )[0].strip()

    return value


# ------------------------------------------------------------
# 본문 안의 bracket 필드 추출
# ------------------------------------------------------------
# 예:
# [서비스명] 청년월세 지원사업
# [지원내용] 월세 일부 지원
# ------------------------------------------------------------

def extract_bracket_field(text: str, labels: List[str]) -> str:
    """
    [서비스명] ..., [지원내용] ... 같은 bracket 필드를 추출한다.
    다음 [필드명] 전까지만 가져온다.
    """
    if not text:
        return ""

    for label in labels:
        pattern = rf"\[{re.escape(label)}\]\s*(.*?)(?=\s*\[[^\]]+\]|$)"
        match = re.search(pattern, text, flags=re.DOTALL)

        if match:
            value = clean_extracted_value(match.group(1))
            if value:
                return value

    return ""


# ------------------------------------------------------------
# 본문 안의 colon 필드 추출
# ------------------------------------------------------------
# 예:
# 정책명: 청년월세 지원사업
# 지원내용: 월세 일부 지원
# ------------------------------------------------------------

def extract_colon_field(text: str, labels: List[str]) -> str:
    """
    정책명: ..., 지원내용: ... 같은 colon 필드를 추출한다.
    """
    if not text:
        return ""

    for label in labels:
        # 한 줄형 필드 우선
        pattern_line = rf"{re.escape(label)}\s*[:：]\s*([^\n\r]+)"
        match = re.search(pattern_line, text)

        if match:
            value = clean_extracted_value(match.group(1))
            if value:
                return value

        # 여러 필드가 같은 줄에 붙어 있는 경우 보완
        pattern_until_field = (
            rf"{re.escape(label)}\s*[:：]\s*(.*?)"
            rf"(?=\s+(?:정책명|서비스명|카테고리|내용|상세 URL|URL|지원대상|지원내용|신청 방법|신청방법|선정기준)\s*[:：]|$)"
        )
        match = re.search(pattern_until_field, text, flags=re.DOTALL)

        if match:
            value = clean_extracted_value(match.group(1))
            if value:
                return value

    return ""


def extract_field_from_text(text: str, labels: List[str]) -> str:
    """
    colon 형식과 bracket 형식을 모두 지원한다.
    """
    value = extract_colon_field(text, labels)
    if value:
        return value

    value = extract_bracket_field(text, labels)
    if value:
        return value

    return ""


# ------------------------------------------------------------
# 정책명 추출
# ------------------------------------------------------------

def extract_policy_name(payload: Dict[str, Any], raw_text: str) -> Tuple[str, str, int]:
    """
    정책명을 추출한다.

    반환값:
    - policy_name: 추출된 정책명
    - source: 어디서 추출했는지
    - priority: 낮을수록 신뢰도 높음
    """
    payload_name = get_payload_value(
        payload,
        "policy_name",
        "title",
        "name",
        "service_name",
        "servNm",
        "정책명",
        "서비스명",
        "제목",
    )

    if payload_name:
        return clean_extracted_value(payload_name), "payload", 0

    colon_policy = extract_colon_field(raw_text, ["정책명", "제목"])
    if colon_policy:
        return colon_policy, "policy_colon", 1

    bracket_policy = extract_bracket_field(raw_text, ["정책명", "제목"])
    if bracket_policy:
        return bracket_policy, "policy_bracket", 1

    colon_service = extract_colon_field(raw_text, ["서비스명"])
    if colon_service:
        return colon_service, "service_colon", 2

    bracket_service = extract_bracket_field(raw_text, ["서비스명"])
    if bracket_service:
        return bracket_service, "service_bracket", 2

    return "정책명 확인 필요", "missing", 9


# ------------------------------------------------------------
# 정책 핵심 내용 추출
# ------------------------------------------------------------

def extract_policy_content(payload: Dict[str, Any], raw_text: str) -> Tuple[str, str]:
    """
    모델과 reranker에 사용할 핵심 내용을 추출한다.
    raw_text 전체보다 지원대상/지원내용/신청정보 중심으로 정리하는 것이 목표다.
    """
    # 1순위: 명시적 내용 필드
    content = extract_colon_field(raw_text, ["내용"])
    if content:
        return content, "content_colon"

    # 2순위: [지원내용]
    content = extract_bracket_field(raw_text, ["지원내용"])
    if content:
        return content, "support_content_bracket"

    # 3순위: 지원내용:
    content = extract_colon_field(raw_text, ["지원내용", "지원 내용"])
    if content:
        return content, "support_content_colon"

    # 4순위: payload summary/content 계열
    payload_content = get_payload_value(
        payload,
        "summary",
        "servDgst",
        "description",
        "desc",
        "content",
        "내용",
        "지원내용",
    )

    if payload_content and payload_content != raw_text:
        return normalize_policy_text(payload_content), "payload_content"

    # 5순위: 지원대상 + 지원내용 + 신청정보 조합
    parts = []

    target = extract_field_from_text(raw_text, ["지원대상", "지원 대상"])
    support = extract_field_from_text(raw_text, ["지원내용", "지원 내용"])
    apply_info = extract_field_from_text(raw_text, ["신청정보", "신청방법", "신청 방법"])

    if target:
        parts.append(f"지원대상: {target}")

    if support:
        parts.append(f"지원내용: {support}")

    if apply_info:
        parts.append(f"신청정보: {apply_info}")

    if parts:
        return " / ".join(parts), "combined_fields"

    # 6순위: 어쩔 수 없을 때 raw text 사용
    return normalize_policy_text(raw_text), "raw_text"


# ------------------------------------------------------------
# URL / 지역 / 섹션 추출
# ------------------------------------------------------------

def extract_url_from_text(text: str) -> str:
    """
    payload에 URL이 없을 때 본문에서 URL을 찾는다.
    """
    if not text:
        return ""

    explicit = extract_field_from_text(text, ["상세 URL", "신청 링크", "URL", "url"])
    if explicit:
        return explicit

    match = re.search(r"https?://[^\s\]]+", text)
    if match:
        return match.group(0).strip()

    return ""


def extract_region(payload: Dict[str, Any], raw_text: str) -> str:
    """
    지역 정보를 추출한다.
    """
    region = get_payload_value(
        payload,
        "region",
        "area",
        "지역",
        "지원 범위",
        "필터링",
    )

    if not region:
        region = extract_field_from_text(raw_text, ["지역", "지원 범위", "필터링"])

    region = clean_extracted_value(region)
    region = re.sub(r"^지역\s+", "", region).strip()
    return region


def extract_section(payload: Dict[str, Any], raw_text: str) -> str:
    """
    정책 분류/섹션 정보를 추출한다.
    """
    section = get_payload_value(
        payload,
        "section",
        "category",
        "카테고리",
        "분류",
        "분야",
        "질문 유형",
        "설명 구분",
        "섹션",
    )

    if not section:
        section = extract_field_from_text(
            raw_text,
            ["카테고리", "분류", "분야", "질문 유형", "설명 구분", "섹션"]
        )

    return clean_extracted_value(section)


# ------------------------------------------------------------
# 검색 결과 1개를 표준 dict로 변환
# ------------------------------------------------------------

def normalize_qdrant_hit(hit: Any, collection_name: str) -> Dict[str, Any]:
    """
    Qdrant 검색 결과 hit 하나를 우리 RAG 파이프라인에서 사용할 표준 형태로 변환한다.
    """
    payload = getattr(hit, "payload", None) or {}
    score = getattr(hit, "score", 0.0)
    point_id = getattr(hit, "id", "")

    raw_text = get_payload_value(
        payload,
        "text",
        "chunk_text",
        "content",
        "page_content",
    )
    raw_text = normalize_policy_text(raw_text)

    policy_name, name_source, name_priority = extract_policy_name(payload, raw_text)
    content, content_source = extract_policy_content(payload, raw_text)

    url = get_payload_value(
        payload,
        "url",
        "source_url",
        "link",
        "servDtlLink",
        "상세 URL",
        "신청 링크",
    )

    if not url:
        url = extract_url_from_text(raw_text)

    region = extract_region(payload, raw_text)
    section = extract_section(payload, raw_text)

    content = normalize_policy_text(content)

    # 내용이 없으면 이후 단계에서 제외할 수 있도록 빈 dict 반환
    if not content:
        return {}

    # 정책명이 잘 잡힌 문서를 약간 우대하기 위한 보정값
    if name_priority <= 1:
        priority_bonus = 0.02
    elif name_priority == 2:
        priority_bonus = 0.0
    else:
        priority_bonus = -0.03

    score_value = float(score) if score is not None else 0.0

    return {
        "point_id": str(point_id),
        "collection": collection_name,
        "score": score_value,
        "adjusted_score": score_value + priority_bonus,
        "policy_name": policy_name,
        "name_source": name_source,
        "name_priority": name_priority,
        "content_source": content_source,
        "text": content,
        "raw_text": raw_text,
        "url": url,
        "region": region,
        "section": section,
    }


# ------------------------------------------------------------
#  Reranker 입력 문서 구성
# ------------------------------------------------------------

def build_reranker_document_text(item: Dict[str, Any]) -> str:
    """
    Reranker가 질문과 비교할 문서 텍스트를 구성한다.

    Reranker는 질문-문서 pair를 보고 관련성을 판단하므로,
    정책명/지역/구분/핵심 내용을 함께 넣어주는 것이 좋다.
    """
    parts = []

    if item.get("policy_name"):
        parts.append(f"정책명: {item['policy_name']}")

    if item.get("region"):
        parts.append(f"지역: {item['region']}")

    if item.get("section"):
        parts.append(f"구분: {item['section']}")

    if item.get("text"):
        parts.append(f"내용: {item['text']}")

    return "\n".join(parts).strip()


print("Payload 정제 함수 준비 완료")

Payload 정제 함수 준비 완료


In [ ]:
# ============================================================
# 10. Qdrant 1차 후보 검색 함수
# ============================================================
# 이 셀은 사용자 질문을 embedding한 뒤,
# Qdrant에서 관련 정책 문서 후보를 넉넉히 가져온다.
#
# [RAG 설계상 역할]
# - 이 단계는 2-stage RAG 중 1단계 dense retrieval이다.
# - 목표는 "정답 문서를 놓치지 않도록" 후보를 넓게 가져오는 것이다.
# - 최종 선택은 다음 단계의 Reranker가 담당한다.
#
# - BAAI/bge-m3: 사용자 질문을 dense vector로 변환
# - Qdrant: 정책 문서 vector search 수행
#
# - 중앙정부/지자체 컬렉션을 모두 검색한다.
# - 각 Qdrant hit를 9번 셀의 normalize_qdrant_hit()으로 표준 형태로 정리한다.
# - 이후 reranker가 사용할 후보 목록을 만든다.
# ============================================================

from typing import Optional


# ------------------------------------------------------------
# Qdrant query wrapper
# ------------------------------------------------------------
# qdrant-client 버전에 따라 query_points() 또는 search()를 사용한다.
# 최신 버전에서는 query_points()를 우선 사용하고,
# 구버전 호환을 위해 search() fallback을 둔다.
# ------------------------------------------------------------

def qdrant_dense_search(
    collection_name: str,
    query_vector: list,
    limit: int,
) -> list:
    """
    Qdrant 컬렉션 하나에서 dense vector search를 수행한다.

    반환:
    - Qdrant hit 목록
    """
    # query_points()를 지원하는 qdrant-client 버전
    if hasattr(qdrant_client, "query_points"):
        kwargs = {
            "collection_name": collection_name,
            "query": query_vector,
            "limit": limit,
            "with_payload": True,
        }

        # named vector 컬렉션을 사용하는 경우
        if QDRANT_VECTOR_NAME:
            kwargs["using"] = QDRANT_VECTOR_NAME

        result = qdrant_client.query_points(**kwargs)

        # qdrant-client 버전에 따라 points 속성에 결과가 들어감
        return getattr(result, "points", result)

    # 구버전 fallback
    kwargs = {
        "collection_name": collection_name,
        "query_vector": query_vector,
        "limit": limit,
        "with_payload": True,
    }

    if QDRANT_VECTOR_NAME:
        kwargs["vector_name"] = QDRANT_VECTOR_NAME

    return qdrant_client.search(**kwargs)


# ------------------------------------------------------------
# 질문 embedding 생성
# ------------------------------------------------------------

def encode_query(query: str) -> list:
    """
    사용자 질문을 Qdrant 검색용 dense vector로 변환한다.
    """
    query_vector = embed_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True,
    )[0].astype("float32").tolist()

    return query_vector


# ------------------------------------------------------------
# 1차 후보 검색
# ------------------------------------------------------------

def retrieve_candidates(query: str, fetch_k: Optional[int] = None) -> list:
    """
    사용자 질문에 대해 Qdrant에서 후보 문서를 넉넉히 검색한다.

    흐름:
    1. 질문을 bge-m3 embedding으로 변환
    2. 중앙정부/지자체 컬렉션을 각각 검색
    3. 검색 결과 hit를 normalize_qdrant_hit()으로 정제
    4. 후보 목록을 반환

    이 단계에서는 최종 top-k를 고르지 않는다.
    최종 재정렬은 다음 단계의 reranker가 담당한다.
    """
    if fetch_k is None:
        fetch_k = QDRANT_FETCH_K

    query = (query or "").strip()

    if not query:
        return []

    query_vector = encode_query(query)

    candidates = []

    print("\n========== Qdrant 1차 후보 검색 ==========")
    print("query:", query)
    print("fetch_k per collection:", fetch_k)
    print("collections:", QDRANT_COLLECTIONS)

    for collection_name in QDRANT_COLLECTIONS:
        try:
            hits = qdrant_dense_search(
                collection_name=collection_name,
                query_vector=query_vector,
                limit=fetch_k,
            )

            print(f"\n[COLLECTION] {collection_name}")
            print(f"raw hits: {len(hits)}")

            for hit in hits:
                item = normalize_qdrant_hit(hit, collection_name)

                # payload 정제 실패 또는 내용 없음
                if not item:
                    continue

                candidates.append(item)

        except Exception as e:
            print(f"[ERROR] Qdrant 검색 실패: {collection_name}")
            print(f"원인: {e}")
            # 한 컬렉션 실패가 전체 검색 실패로 이어지는 게 맞는지 고민할 수 있지만,
            # 현재는 오류를 명확히 보기 위해 raise한다.
            raise

    print("\n정제된 후보 수:", len(candidates))

    # 1차 검색 점수 기준으로 임시 정렬
    candidates.sort(
        key=lambda x: x.get("adjusted_score", x.get("score", 0.0)),
        reverse=True,
    )

    # 디버그 출력: 상위 후보 몇 개만 확인
    print("\n[1차 검색 상위 후보]")
    for i, item in enumerate(candidates[:10], 1):
        print(
            f"{i}. [{item.get('collection')}] "
            f"{item.get('policy_name')} | "
            f"score={item.get('score'):.4f} | "
            f"adjusted={item.get('adjusted_score'):.4f} | "
            f"region={item.get('region') or '-'}"
        )

    print("==========================================\n")

    return candidates


print("Qdrant 1차 후보 검색 함수 준비 완료")

Qdrant 1차 후보 검색 함수 준비 완료


In [ ]:
# ============================================================
# 11. Reranker 재정렬 함수
# ============================================================
# 이 셀은 Qdrant 1차 검색 후보를 Reranker로 다시 정렬한다.
#
# [RAG 설계상 역할]
# - Qdrant dense retrieval은 빠르게 후보를 넓게 가져오는 단계이다.
# - Reranker는 질문과 후보 문서를 pair로 함께 보고 관련성 점수를 다시 계산한다.
# - 최종 LLM context에는 Reranker 점수가 높은 문서를 우선 사용한다.
#
# [외부 공개 모델 활용]
# - Dongjin-kr/ko-reranker
# - 역할: 질문-문서 pair의 관련성 점수 계산
#
# - Qdrant hit에서 추출한 정책명/지역/구분/내용을 조합해
#   Reranker 입력용 문서 텍스트를 구성한다.
# - Reranker 점수를 각 후보 item에 reranker_score로 저장한다.
# ============================================================

import math
import torch
from typing import List, Dict, Any


# ------------------------------------------------------------
#  Reranker 점수 계산
# ------------------------------------------------------------

def compute_reranker_scores(
    query: str,
    candidates: List[Dict[str, Any]],
    batch_size: int = 8,
) -> List[float]:
    """
    사용자 질문과 후보 문서 목록을 pair로 만들어 reranker 점수를 계산한다.

    입력:
    - query: 사용자 질문
    - candidates: Qdrant 1차 검색 후보 목록
    - batch_size: reranker 추론 batch 크기

    출력:
    - 후보 문서별 reranker 점수 리스트
    """
    if not candidates:
        return []

    query = (query or "").strip()
    scores = []

    reranker_device = next(reranker_model.parameters()).device

    for start in range(0, len(candidates), batch_size):
        batch = candidates[start:start + batch_size]

        # Reranker에 넣을 질문-문서 pair 구성
        query_texts = []
        doc_texts = []

        for item in batch:
            doc_text = build_reranker_document_text(item)
            query_texts.append(query)
            doc_texts.append(doc_text)

        inputs = reranker_tokenizer(
            query_texts,
            doc_texts,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt",
        )

        inputs = {
            key: value.to(reranker_device)
            for key, value in inputs.items()
        }

        with torch.no_grad():
            outputs = reranker_model(**inputs)

            # 일반적인 reranker는 logits가 [batch, 1] 또는 [batch] 형태로 나온다.
            logits = outputs.logits

            if logits.ndim == 2 and logits.shape[1] == 1:
                batch_scores = logits.squeeze(-1)
            elif logits.ndim == 2 and logits.shape[1] > 1:
                # 혹시 2-class classification 형태라면 positive class 점수를 사용
                batch_scores = logits[:, -1]
            else:
                batch_scores = logits.view(-1)

        scores.extend(batch_scores.detach().float().cpu().tolist())

    return scores


# ------------------------------------------------------------
# 후보 목록 재정렬
# ------------------------------------------------------------

def rerank_candidates(
    query: str,
    candidates: List[Dict[str, Any]],
    top_n: int = None,
) -> List[Dict[str, Any]]:
    """
    Qdrant 1차 후보를 Reranker 점수 기준으로 재정렬한다.

    흐름:
    1. 각 후보 문서에 대해 질문-문서 pair 생성
    2. Dongjin-kr/ko-reranker로 관련성 점수 계산
    3. item["reranker_score"]에 점수 저장
    4. reranker_score 기준 내림차순 정렬
    5. 상위 top_n개 반환
    """
    if top_n is None:
        top_n = RERANK_TOP_N

    if not candidates:
        return []

    print("\n========== Reranker 재정렬 ==========")
    print("query:", query)
    print("input candidates:", len(candidates))
    print("top_n:", top_n)

    scores = compute_reranker_scores(
        query=query,
        candidates=candidates,
        batch_size=8,
    )

    reranked = []

    for item, score in zip(candidates, scores):
        new_item = dict(item)
        new_item["reranker_score"] = float(score)
        reranked.append(new_item)

    # Reranker 점수 기준 정렬
    reranked.sort(
        key=lambda x: x.get("reranker_score", -math.inf),
        reverse=True,
    )

    # 상위 top_n만 반환
    reranked = reranked[:top_n]

    print("\n[Reranker 상위 후보]")
    for i, item in enumerate(reranked[:10], 1):
        print(
            f"{i}. [{item.get('collection')}] "
            f"{item.get('policy_name')} | "
            f"reranker={item.get('reranker_score'):.4f} | "
            f"qdrant={item.get('score'):.4f} | "
            f"region={item.get('region') or '-'}"
        )

    print("=====================================\n")

    return reranked


print("Reranker 재정렬 함수 준비 완료")

Reranker 재정렬 함수 준비 완료


In [ ]:
# ============================================================
# 12. 최종 LLM context 구성 함수
# ============================================================
# 이 셀은 Reranker가 재정렬한 후보 문서 중에서
# 최종적으로 LLM에게 제공할 정책 문서 context를 만든다.
#
# [RAG 설계상 역할]
# - 10번 셀: Qdrant 1차 후보 검색
# - 11번 셀: Dongjin-kr/ko-reranker로 질문 관련성 재정렬
# - 12번 셀: 중복 정책 제거 + 최종 문서 선택 + LLM context 구성
#
# - 같은 정책이 여러 chunk로 검색될 수 있으므로 중복을 줄인다.
# - Reranker 점수가 높은 정책을 우선 사용한다.
# - Qwen에게 너무 긴 문서를 넣지 않도록 정책별 길이를 제한한다.
# - 정책명/지역/구분/내용을 정리해 LLM이 읽기 쉬운 context로 만든다.
# ============================================================

from typing import List, Dict, Any


# ------------------------------------------------------------
# 중복 정책 제거
# ------------------------------------------------------------
# Qdrant는 chunk 단위로 검색하므로 같은 정책의 여러 조각이 검색될 가능성 미연에 방지
# 동일 정책이 context를 과도하게 차지하지 않도록 정책명+URL 기준으로 묶는다.
# ------------------------------------------------------------

def group_chunks_by_policy(items: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """
    reranker로 정렬된 후보들을 정책 단위로 묶는다.

    반환:
    - 정책별 대표 정보와 관련 chunk 목록을 가진 리스트
    """
    grouped = {}

    for item in items:
        policy_name = item.get("policy_name") or "정책명 확인 필요"
        url = item.get("url") or ""

        # 정책명과 URL이 같으면 같은 정책으로 간주
        # URL이 비어 있더라도 정책명이 같으면 우선 같은 정책으로 묶인다.
        group_key = (policy_name, url)

        if group_key not in grouped:
            grouped[group_key] = {
                "policy_name": policy_name,
                "url": url,
                "collection": item.get("collection", ""),
                "region": item.get("region", ""),
                "section": item.get("section", ""),
                "best_qdrant_score": item.get("score", 0.0),
                "best_reranker_score": item.get("reranker_score", 0.0),
                "chunks": [],
            }

        grouped_item = grouped[group_key]

        # 같은 정책 안에서는 가장 높은 점수를 보존
        grouped_item["best_qdrant_score"] = max(
            grouped_item.get("best_qdrant_score", 0.0),
            item.get("score", 0.0),
        )

        grouped_item["best_reranker_score"] = max(
            grouped_item.get("best_reranker_score", 0.0),
            item.get("reranker_score", 0.0),
        )

        # region/section이 비어 있으면 후속 chunk에서 보완
        if not grouped_item.get("region") and item.get("region"):
            grouped_item["region"] = item.get("region")

        if not grouped_item.get("section") and item.get("section"):
            grouped_item["section"] = item.get("section")

        grouped_item["chunks"].append(item)

    # Reranker 점수가 높은 정책 순으로 정렬
    grouped_list = list(grouped.values())
    grouped_list.sort(
        key=lambda x: x.get("best_reranker_score", 0.0),
        reverse=True,
    )

    return grouped_list


# ------------------------------------------------------------
# 정책별 context 텍스트 구성
# ------------------------------------------------------------
# LLM에게 그대로 raw text를 넣지 않고,
# 정책명/지역/구분/내용 중심으로 정리해서 넣는다.
# ------------------------------------------------------------

def build_policy_context_text(policy_item: Dict[str, Any]) -> str:
    """
    정책 하나에 대해 Qwen에게 넣을 context 텍스트를 구성한다.
    """
    policy_name = policy_item.get("policy_name", "정책명 확인 필요")
    region = policy_item.get("region", "")
    section = policy_item.get("section", "")
    chunks = policy_item.get("chunks", [])

    # 같은 정책의 여러 chunk 내용을 이어붙임
    chunk_texts = []
    seen_texts = set()

    for chunk in chunks:
        text = (chunk.get("text") or "").strip()

        if not text:
            continue

        # 완전히 같은 내용 또는 거의 같은 내용 중복 방지
        normalized = text[:200]

        if normalized in seen_texts:
            continue

        seen_texts.add(normalized)
        chunk_texts.append(text)

    joined_text = " ".join(chunk_texts).strip()

    # 정책별 context 길이 제한
    # 너무 긴 정책 문서가 전체 prompt를 과도하게 차지하지 않도록 제한한다.
    if len(joined_text) > MAX_CONTEXT_CHARS_PER_POLICY:
        joined_text = joined_text[:MAX_CONTEXT_CHARS_PER_POLICY].rstrip() + "..."

    parts = [
        f"정책명: {policy_name}",
    ]

    if region:
        parts.append(f"지역: {region}")

    if section:
        parts.append(f"구분: {section}")

    if joined_text:
        parts.append(f"내용: {joined_text}")

    return "\n".join(parts).strip()


# ------------------------------------------------------------
# 최종 LLM context 구성
# ------------------------------------------------------------
# Reranker가 재정렬한 후보 중 최종 top-k 정책만 골라
# Qwen에게 제공할 [문서] context를 만든다.
# ------------------------------------------------------------

def build_final_context(
    reranked_items: List[Dict[str, Any]],
    final_top_k: int = None,
) -> Dict[str, Any]:
    """
    Reranker가 정렬한 후보 문서를 최종 LLM context로 변환한다.

    반환:
    {
        "context_text": LLM에 넣을 문서 문자열,
        "selected_policies": 최종 선택된 정책 목록
    }
    """
    if final_top_k is None:
        final_top_k = FINAL_TOP_K

    if not reranked_items:
        return {
            "context_text": "검색된 관련 정책 문서가 없습니다.",
            "selected_policies": [],
        }

    # 같은 정책의 여러 chunk를 하나로 묶음
    grouped_policies = group_chunks_by_policy(reranked_items)

    # 최종 top-k 정책만 사용
    selected_policies = grouped_policies[:final_top_k]

    context_blocks = []

    for idx, policy in enumerate(selected_policies, 1):
        policy_text = build_policy_context_text(policy)

        if policy_text:
            context_blocks.append(
                f"[문서 {idx}]\n{policy_text}"
            )

    context_text = "\n\n".join(context_blocks).strip()

    if not context_text:
        context_text = "검색된 관련 정책 문서가 없습니다."

    # 디버깅용 로그
    # 답변 화면에는 참고문서 블록을 붙이지 않지만,
    # Colab 로그에서는 어떤 정책이 최종 선택되었는지 확인할 수 있게 한다.
    print("\n========== 최종 LLM context 구성 ==========")
    print("최종 선택 정책 수:", len(selected_policies))

    for i, policy in enumerate(selected_policies, 1):
        print(
            f"{i}. {policy.get('policy_name')} | "
            f"reranker={policy.get('best_reranker_score'):.4f} | "
            f"region={policy.get('region') or '-'} | "
            f"collection={policy.get('collection') or '-'}"
        )

    print("==========================================\n")

    return {
        "context_text": context_text,
        "selected_policies": selected_policies,
    }


print("최종 context 구성 함수 준비 완료")

최종 context 구성 함수 준비 완료


In [ ]:
# ============================================================
# 13. RAG 기반 답변 생성 함수
# ============================================================
# 이 셀은 /ask endpoint에서 실제로 호출할 답변 생성 함수를 정의한다.
#
# [전체 RAG 흐름]
# 1. 사용자 질문 입력
# 2. Qdrant dense retrieval로 후보 문서 검색
# 3. Dongjin-kr/ko-reranker로 후보 문서 재정렬
# 4. 최종 LLM context 구성
# 5. LLM이 답변 생성
#
# [외부 공개 모델 활용]
# - Qwen: 최종 답변 생성
# - BAAI/bge-m3: Qdrant 검색용 질문 embedding
# - Dongjin-kr/ko-reranker: 후보 문서 재정렬
#
# - 정책 상담용 system prompt 작성
# - 검색 문서 context 구성
# - 문서 기반 답변 원칙 적용
# - 참고문서 블록은 답변에 붙이지 않음
# ============================================================


# ------------------------------------------------------------
#정책 상담용 system prompt
# ------------------------------------------------------------
# 모델이 검색 문서를 바탕으로 자연스럽게 답변하도록 제어한다.
# ------------------------------------------------------------

SYSTEM_PROMPT = """
너는 대한민국 청년 정책을 안내하는 친절하고 정확한 정책 상담사야.

반드시 제공된 [문서] 내용만을 근거로 답변해.
문서에 없는 내용은 추측하지 말고, 정보가 부족하다고 말해.
DB 필드명을 그대로 나열하지 말고, 사람이 이해하기 쉬운 자연스러운 문장으로 답변해.
사용자의 상황과 가장 관련 있는 정책을 중심으로 설명해.
반드시 한국어로만 답변해.

답변은 너무 짧게 끝내지 말고, 사용자가 실제로 이해하고 다음 행동을 정할 수 있도록 충분히 설명해.
다만 문서에 없는 신청 자격, 금액, 기간, 링크, 절차를 임의로 만들어내지 마.

답변 구조는 다음을 따르는 것을 권장해.
1. 사용자의 상황에 맞는 정책이 있는지 먼저 요약
2. 관련 정책이 있다면 정책명과 지원 내용을 설명
3. 지원 대상이나 조건이 문서에 있으면 함께 설명
4. 신청 방법이나 확인해야 할 사항이 문서에 있으면 안내
5. 문서에 부족한 정보가 있으면 추가 확인이 필요하다고 말하기

답변은 보통 2~4개 단락으로 작성해.
관련 정책이 여러 개라면 정책별로 나누어 설명해.
""".strip()


# ------------------------------------------------------------
# prompt 구성
# ------------------------------------------------------------

def build_messages_for_generation(query: str, context_text: str) -> list:
    """
    LLM에 전달할 chat message 목록을 구성한다.
    """
    user_content = f"""
[문서]
{context_text}

[사용자 질문]
{query}
""".strip()

    return [
        {
            "role": "system",
            "content": SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": user_content,
        },
    ]


# ------------------------------------------------------------
# LLM 답변 생성
# ------------------------------------------------------------

def generate_with_qwen(query: str, context_text: str) -> str:
    """
    최종 context와 사용자 질문을 바탕으로 답변을 생성한다.
    """
    import time #tps계산용
    messages = build_messages_for_generation(query, context_text)

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    )

    inputs = {
        key: value.to(model.device)
        for key, value in inputs.items()
    }

    #tps계산용
    if torch.cuda.is_available():
      torch.cuda.synchronize()

    start_time = time.perf_counter()
    #


    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=True,
            temperature=GEN_TEMPERATURE,
            top_p=GEN_TOP_P,
            repetition_penalty=GEN_REPETITION_PENALTY,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            return_dict_in_generate=True,
        )

    #tps계산용
    if torch.cuda.is_available():
      torch.cuda.synchronize()

    end_time = time.perf_counter()
    generation_seconds = end_time - start_time
    #

    input_length = inputs["input_ids"].shape[1]
    generated_ids = outputs.sequences[0][input_length:]

    #tps계산용
    generated_token_count = int(generated_ids.shape[0])
    tps = generated_token_count / generation_seconds if generation_seconds > 0 else 0.0

    print("\n========== 생성 속도 측정 ==========")
    print(f"생성 token 수  : {generated_token_count}")
    print(f"생성 시간      : {generation_seconds:.3f} sec")
    print(f"TPS            : {tps:.2f} tokens/sec")
    print("====================================\n")
    #

    answer = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True,
    ).strip()

    return answer


# ------------------------------------------------------------
# 전체 RAG 답변 생성 함수
# ------------------------------------------------------------

def generate_rag_answer(query: str) -> str:
    """
    사용자 질문에 대해 RAG 기반 정책 상담 답변을 생성한다.

    흐름:
    1. Qdrant 1차 후보 검색
    2. Dongjin-kr/ko-reranker 재정렬
    3. 최종 LLM context 구성
    4. LLM 답변 생성
    """
    query = (query or "").strip()

    if not query:
        return "질문 내용이 비어 있습니다. 궁금한 정책이나 상황을 입력해 주세요."

    print("\n\n==================================================")
    print("[RAG 답변 생성 시작]")
    print("질문:", query)
    print("==================================================")

    # 1. Qdrant 1차 후보 검색
    candidates = retrieve_candidates(query)

    if not candidates:
        print("[WARN] Qdrant 검색 결과가 없습니다.")
        return (
            "현재 질문과 직접 관련된 정책 문서를 찾지 못했습니다. "
            "지역, 나이, 현재 상태를 조금 더 구체적으로 적어주시면 다시 확인해볼 수 있습니다."
        )

    # 2. ko-reranker 재정렬
    reranked_items = rerank_candidates(query, candidates)

    if not reranked_items:
        print("[WARN] Reranker 재정렬 결과가 없습니다.")
        return (
            "검색된 정책 문서를 재정렬하는 과정에서 관련 문서를 확정하지 못했습니다. "
            "질문을 조금 더 구체적으로 입력해 주세요."
        )

    # 3. 최종 context 구성
    context_result = build_final_context(reranked_items)
    context_text = context_result["context_text"]
    selected_policies = context_result["selected_policies"]

    if not selected_policies:
        print("[WARN] 최종 선택 정책이 없습니다.")
        return (
            "질문과 관련된 정책 문서를 충분히 찾지 못했습니다. "
            "지역, 나이, 취업 여부 등 조건을 함께 입력해 주세요."
        )

    # 4. LLM 답변 생성
    answer = generate_with_qwen(query, context_text)

    if not answer:
        answer = (
            "답변을 생성하지 못했습니다. "
            "질문을 조금 더 구체적으로 입력해 주세요."
        )

    # 참고문서 블록은 붙이지 않는다.
    # 단, 최종 선택 정책은 Colab 로그에서 확인 가능하다.
    print("[RAG 답변 생성 완료]")
    print("선택 정책 수:", len(selected_policies))
    print("답변:", answer[:500])
    print("==================================================\n")

    return answer


print("RAG 답변 생성 함수 준비 완료")

RAG 답변 생성 함수 준비 완료


In [ ]:
# ============================================================
# 14. Colab 내부 RAG 테스트
# 정상적으로 동작하는지 확인하기 위한 테스트 셀
# ============================================================

test_query = "서울 거주 25세 취업준비생인데 면접정장 대여 지원 있어?"

test_answer = generate_rag_answer(test_query)

print("\n========== 테스트 질문 ==========")
print(test_query)

print("\n========== 생성 답변 ==========")
print(test_answer)



[RAG 답변 생성 시작]
질문: 서울 거주 25세 취업준비생인데 면접정장 대여 지원 있어?

========== Qdrant 1차 후보 검색 ==========
query: 서울 거주 25세 취업준비생인데 면접정장 대여 지원 있어?
fetch_k per collection: 30
collections: ['welfare_ministry_bge_m3', 'local_government_welfare_bge_m3']

[COLLECTION] welfare_ministry_bge_m3
raw hits: 30

[COLLECTION] local_government_welfare_bge_m3
raw hits: 30

정제된 후보 수: 60

[1차 검색 상위 후보]
1. [local_government_welfare_bge_m3] 청년 면접정장 무료대여사업 | score=0.7259 | adjusted=0.7459 | region=경기도 과천시
2. [local_government_welfare_bge_m3] 구직청년 면접용 정장등 대여 | score=0.7160 | adjusted=0.7360 | region=대전광역시
3. [local_government_welfare_bge_m3] 청년 면접정장 대여사업 | score=0.7112 | adjusted=0.7312 | region=경기도 부천시
4. [local_government_welfare_bge_m3] 청년면접정장 대여 사업 | score=0.7080 | adjusted=0.7280 | region=울산광역시 중구
5. [local_government_welfare_bge_m3] 청년구직자 면접정장 대여사업 | score=0.6961 | adjusted=0.7161 | region=경기도 구리시
6. [local_government_welfare_bge_m3] 청년 면접정장 무료대여 지원사업 | score=0.6924 | adjusted=0.7124 | region=전북특별자치도 전주시
7. [local

In [ ]:
test_query = "서울 거주 25세 대학생인데 자격증 취득 지원 등의 취업 준비를 돕는 정책을 알려줘"

test_answer = generate_rag_answer(test_query)

print("\n========== 테스트 질문 ==========")
print(test_query)

print("\n========== 생성 답변 ==========")
print(test_answer)



[RAG 답변 생성 시작]
질문: 서울 거주 25세 대학생인데 자격증 취득 지원 등의 취업 준비를 돕는 정책을 알려줘

========== Qdrant 1차 후보 검색 ==========
query: 서울 거주 25세 대학생인데 자격증 취득 지원 등의 취업 준비를 돕는 정책을 알려줘
fetch_k per collection: 30
collections: ['welfare_ministry_bge_m3', 'local_government_welfare_bge_m3']

[COLLECTION] welfare_ministry_bge_m3
raw hits: 30

[COLLECTION] local_government_welfare_bge_m3
raw hits: 30

정제된 후보 수: 60

[1차 검색 상위 후보]
1. [local_government_welfare_bge_m3] 청년 취업 자격취득 활동 지원사업 | score=0.6256 | adjusted=0.6456 | region=서울특별시 동대문구
2. [local_government_welfare_bge_m3] 청년 자격증 등 응시료 지원 | score=0.6190 | adjusted=0.6390 | region=서울특별시 중구
3. [local_government_welfare_bge_m3] 제주시 취업준비 청년 자격증 등 응시료 지원사업 | score=0.6147 | adjusted=0.6347 | region=제주특별자치도 제주시
4. [local_government_welfare_bge_m3] 저소득층 대학생 교육비 지원사업 | score=0.6086 | adjusted=0.6286 | region=서울특별시 강남구
5. [local_government_welfare_bge_m3] 강남구 미취업 청년 어학·자격증 응시료 지원 사업 | score=0.6082 | adjusted=0.6282 | region=서울특별시 강남구
6. [local_government_welfare_bge_m3] 서울런(S